# NeoRX · Evaluación completa con CPU de Kaggle (sin cuota GPU)
Utiliza este notebook **cuando termine la cuarta sesión** y hayas guardado su salida.
Adjunta el paquete ORIGINAL `neorx_kaggle_entrenamiento`, NIH completo y el avance más reciente.
No vuelvas a subir ni cambies el ZIP de código. Esta evaluación conserva los pesos y umbrales de `best.pt`.
El cierre por presupuesto queda registrado sin afirmar que se completaron las 21 épocas.
Selecciona **Accelerator: None / CPU**, activa Internet y usa **Save Version → Save & Run All**. Se evalúa todo TEST con el mismo modelo, resolución y umbrales. CPU puede tardar varias horas o superar el límite de sesión: no hay duración garantizada. El log estima el tiempo restante de inferencia una vez que empieza a procesar imágenes.
Después de consultar TEST, no lo uses para decidir otras épocas, modelos o umbrales: necesitarías otro conjunto independiente.

In [ ]:
# Completa con la ruta REAL del avance guardado al terminar la cuarta sesión.
PREVIOUS_OUTPUT = "/kaggle/input/datasets/javier64/neorx-avance-4/neorx-run"  # /kaggle/input/.../neorx-run
DATA_ROOT = ""        # detectar automáticamente la entrada NIH
CLOSE_BY_BUDGET = True  # cierre definitivo de este experimento por tiempo disponible

In [ ]:
from pathlib import Path
import json, os, sys, zipfile, subprocess, shutil

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
bundles = list(INPUT.rglob('bundle_manifest.json'))
if not bundles:
    archives = list(INPUT.rglob('neorx_kaggle_entrenamiento.zip'))
    if len(archives) != 1:
        raise RuntimeError('Agrega el paquete de código como Dataset privado; debe haber una sola copia.')
    target = WORK / 'neorx-code'
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            resolved = (target / member.filename).resolve()
            if not resolved.is_relative_to(target.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('Ruta no permitida en el ZIP')
        z.extractall(target)
    bundles = list(target.rglob('bundle_manifest.json'))
if len(bundles) != 1:
    raise RuntimeError('Hay varios paquetes de código. Deja solo la versión que vas a utilizar.')
BUNDLE = bundles[0].parent
print('Código:', BUNDLE)
print('Entradas NIH:', [str(p.parent) for p in INPUT.rglob('Data_Entry_2017.csv')])
print('Salidas anteriores:', [str(p.parent) for p in INPUT.rglob('STATUS.json')])
# PyTorch/CUDA se conservan tal como los proporciona Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                '-r', str(BUNDLE / 'training/requirements.kaggle.txt')], check=True)
sys.path.insert(0, str(BUNDLE))
os.environ['PYTHONPATH'] = str(BUNDLE)
os.environ['PYTHONUNBUFFERED'] = '1'
from training.kaggle_run import verify_bundle
info = verify_bundle(BUNDLE)
import torch
print('Evaluación en CPU de Kaggle | PyTorch:', torch.__version__)
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
print('Imágenes de referencia:', info['reference_images'])

In [ ]:
# Evaluador independiente; no modifica el paquete original.
script_path = WORK / 'evaluate_frozen.py'
script_path.write_text('"""Freeze the validation-selected checkpoint after an explicit budget stop.\n\nThis separate entry point works with the ORIGINAL Kaggle bundle. It never\nalters that bundle, trains, tunes thresholds on TEST or claims 21 epochs ran.\n"""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\n\nimport yaml\n\nfrom training.analyze_targets import analyze\nfrom training.configuration import get_target_labels\nfrom training.kaggle_run import discover_data_root, package_outputs, verify_bundle, write_json\nfrom training.model import load_checkpoint\nfrom training.session import resume_signature\nfrom training.validate_dataset import index_images\n\n\ndef prepare_evaluation(bundle, previous, output, data_root, close_by_budget):\n    build = verify_bundle(bundle)\n    if json.loads((previous / \'source_bundle.json\').read_text(encoding=\'utf-8\')) != build:\n        raise ValueError(\'Use the ORIGINAL code bundle from training\')\n    if (previous / \'evaluation\').exists():\n        raise ValueError(\'This run already has TEST results; use those results instead of selecting another model\')\n    state = json.loads((previous / \'checkpoints/session_status.json\').read_text(encoding=\'utf-8\'))\n    if not state.get(\'training_finished\'):\n        if state.get(\'status\') != \'paused_time_budget\' or not close_by_budget:\n            raise ValueError(\'Explicit budget closure is required for a paused run\')\n    cfg = yaml.safe_load((previous / \'config.runtime.yaml\').read_text(encoding=\'utf-8\'))\n    manifest_text = (previous / \'manifests/clean_manifest.csv\').read_text(encoding=\'utf-8-sig\')\n    splits = json.loads((previous / \'manifests/splits.json\').read_text(encoding=\'utf-8\'))\n    analyze(manifest_text, splits)\n    folder = output / \'manifests\'\n    folder.mkdir(parents=True)\n    manifest = folder / \'clean_manifest.csv\'\n    manifest.write_text(manifest_text, encoding=\'utf-8\')\n    write_json(folder / \'splits.json\', splits)\n    cfg[\'data\'].update(data_dir=str(data_root), manifest=str(manifest), splits_json=str(folder / \'splits.json\'))\n    last = load_checkpoint(previous / \'checkpoints/last.pt\')\n    if last.get(\'training_state\', {}).get(\'resume_signature\') != resume_signature(cfg):\n        raise ValueError(\'Dataset/configuration identity differs from the saved training run\')\n    completed = int(last[\'training_state\'][\'next_epoch\'])\n    del last\n    best_path = previous / \'checkpoints/best.pt\'\n    best = load_checkpoint(best_path)\n    labels = get_target_labels(cfg)\n    if best.get(\'pathologies\') != labels:\n        raise ValueError(\'Best checkpoint labels differ from training\')\n    if best.get(\'extra\', {}).get(\'config\', {}).get(\'decision_strategy\') != cfg.get(\'decision_strategy\'):\n        raise ValueError(\'Best checkpoint routing policy differs from training\')\n    for label in labels:\n        threshold = best.get(\'thresholds\', {}).get(label)\n        policy = best.get(\'decision_policy\', {}).get(label)\n        if not isinstance(threshold, (int, float)) or not math.isfinite(threshold) or not 0 <= threshold <= 1:\n            raise ValueError(f\'Missing or invalid validation threshold: {label}\')\n        if not isinstance(policy, dict) or not isinstance(policy.get(\'enabled\'), bool):\n            raise ValueError(f\'Missing validation decision policy: {label}\')\n    selected_epoch = best.get(\'extra\', {}).get(\'epoch\')\n    if not isinstance(selected_epoch, int) or not 0 <= selected_epoch < completed:\n        raise ValueError(\'Invalid selected checkpoint epoch\')\n    target = output / \'checkpoints\' / cfg[\'model\'][\'checkpoint_name\']\n    target.parent.mkdir()\n    shutil.copy2(best_path, target)\n    selected_hash = hashlib.sha256(target.read_bytes()).hexdigest()\n    record = {\n        \'status\': \'frozen_before_test\',\n        \'closure_reason\': \'time_budget\' if not state.get(\'training_finished\') else state[\'status\'],\n        \'original_training_status\': state, \'completed_epochs\': completed,\n        \'selected_epoch_zero_based\': selected_epoch, \'selected_by\': \'VALIDATION primary average precision\',\n        \'checkpoint_sha256\': selected_hash, \'thresholds\': best[\'thresholds\'],\n        \'decision_policy\': best[\'decision_policy\'], \'validation_metrics\': best.get(\'metrics\', {}),\n        \'note\': \'No claim of completing all planned epochs. TEST must not guide further model selection. Internal NIH evaluation, not external clinical validation.\',\n    }\n    write_json(output / \'FREEZE.json\', record)\n    write_json(output / \'source_bundle.json\', build)\n    del best\n\n    paths = index_images(data_root)\n    with manifest.open(encoding=\'utf-8\', newline=\'\') as handle:\n        reader = csv.DictReader(handle)\n        fields, rows = reader.fieldnames, list(reader)\n    test_ids = set(splits[\'test\'])\n    if not test_ids or not all(isinstance(name, str) for name in test_ids):\n        raise ValueError(\'Expected the original named TEST partition\')\n    test_rows = []\n    for row in rows:\n        matches = paths.get(row[\'Image Index\'], [])\n        if len(matches) != 1:\n            raise ValueError(f"Missing/ambiguous dataset file: {row[\'Image Index\']}")\n        row[\'image_path\'] = str(matches[0])\n        if row[\'Image Index\'] in test_ids:\n            test_rows.append(row)\n\n    def verify_pixels(row):\n        # Compare TEST bytes against the training audit, not a newly accepted hash.\n        digest = hashlib.sha256(Path(row[\'image_path\']).read_bytes()).hexdigest()\n        if digest != row.get(\'sha256\'):\n            raise ValueError(f"TEST image changed: {row[\'Image Index\']}")\n\n    print(f\'Checking the {len(test_rows):,} TEST images against the original audit; no full image re-audit.\', flush=True)\n    with ThreadPoolExecutor(max_workers=2) as pool:\n        for index, _ in enumerate(pool.map(verify_pixels, test_rows), 1):\n            if index % 1000 == 0:\n                print(f\'TEST verified: {index:,}/{len(test_rows):,}\', flush=True)\n    with manifest.open(\'w\', encoding=\'utf-8\', newline=\'\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    cfg[\'model\'].update(cache_dir=str(bundle / \'weights\'), checkpoint_dir=str(target.parent))\n    runtime = output / \'config.runtime.yaml\'\n    runtime.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding=\'utf-8\')\n    write_json(output / \'input_verification.json\', {\'test_images_verified\': len(test_rows), \'hashes_match_training_audit\': True})\n    return runtime, target\n\n\nclass ProgressLoader:\n    """Report completed inference batches without changing their contents/order."""\n\n    def __init__(self, loader):\n        self.loader = loader\n\n    def __iter__(self):\n        started = time.monotonic()\n        last_report = started\n        completed = 0\n        total = len(self.loader.dataset)\n        for batch in self.loader:\n            yield batch\n            completed += len(batch[2])\n            now = time.monotonic()\n            if completed == len(batch[2]) or now - last_report >= 30 or completed == total:\n                elapsed = max(now - started, 1e-6)\n                rate = completed / elapsed\n                remaining = max(0, total - completed) / rate\n                print(f\'TEST inference: {completed:,}/{total:,} | {rate:.2f} images/s | \'\n                      f\'estimated remaining inference: {remaining / 3600:.2f} h \'\n                      \'(excludes setup, verification and final reports)\', flush=True)\n                last_report = now\n\n\ndef run_evaluation(config, target, output, device, cpu_threads=2):\n    """Use original metrics/inference code, adding only resource limits/progress."""\n    import torch\n    from training import evaluate as evaluator\n\n    if device == \'cpu\':\n        torch.set_num_threads(cpu_threads)\n        cfg = yaml.safe_load(config.read_text(encoding=\'utf-8\'))\n        original_batch = cfg[\'train\'][\'batch_size\']\n        original_workers = cfg[\'train\'].get(\'num_workers\', 0)\n        # Identity was checked before this evaluation-only resource adjustment.\n        cfg[\'train\'].update(batch_size=2, num_workers=2)\n        config.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding=\'utf-8\')\n        write_json(output / \'evaluation_resources.json\', {\n            \'device\': \'cpu\', \'threads\': cpu_threads, \'batch_size\': 2, \'num_workers\': 2,\n            \'original_batch_size\': original_batch, \'original_num_workers\': original_workers,\n            \'full_test_set\': True, \'image_resolution_unchanged\': True,\n        })\n    original_evaluate = evaluator.evaluate_model\n    original_argv = sys.argv\n\n    def with_progress(model, loader, selected_device):\n        return original_evaluate(model, ProgressLoader(loader), selected_device)\n\n    try:\n        evaluator.evaluate_model = with_progress\n        sys.argv = [\'training.evaluate\', \'--config\', str(config), \'--checkpoint\', str(target),\n                    \'--device\', device, \'--output-dir\', str(output / \'evaluation\'), \'--plot\']\n        evaluator.main()\n    finally:\n        sys.argv = original_argv\n        evaluator.evaluate_model = original_evaluate\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--previous-output\', type=Path, required=True)\n    parser.add_argument(\'--data-root\', type=Path)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'/kaggle/working/neorx-evaluation\'))\n    parser.add_argument(\'--close-by-budget\', action=\'store_true\')\n    parser.add_argument(\'--device\', choices=[\'cuda\', \'cpu\'], default=\'cuda\')\n    parser.add_argument(\'--cpu-threads\', type=int, default=2)\n    args = parser.parse_args()\n    if args.cpu_threads < 1:\n        parser.error(\'--cpu-threads must be positive\')\n    output = args.output.resolve()\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(\'Use a fresh Kaggle version; this evaluation output already exists\')\n    output.mkdir(parents=True, exist_ok=True)\n    status = {\'status\': \'failed\', \'test_evaluated\': False}\n    try:\n        import torch\n        if args.device == \'cuda\' and not torch.cuda.is_available():\n            raise RuntimeError(\'Enable GPU in Kaggle\')\n        print(f\'Evaluation device: {args.device}. Full TEST set; no training.\', flush=True)\n        data_root = args.data_root or discover_data_root(Path(\'/kaggle/input\'))\n        cfg, target = prepare_evaluation(args.bundle.resolve(), args.previous_output.resolve(), output,\n                                         data_root.resolve(), args.close_by_budget)\n        run_evaluation(cfg, target, output, args.device, args.cpu_threads)\n        status = {\'status\': \'evaluated\', \'test_evaluated\': True, \'training_run_frozen\': True,\n                  \'note\': \'Review metrics and routing_summary.json before considering deployment; evaluation does not improve weights.\'}\n        print(json.dumps(status, indent=2), flush=True)\n    except BaseException as exc:\n        status[\'error\'] = str(exc)\n        raise\n    finally:\n        write_json(output / \'STATUS.json\', status)\n        package_outputs(output)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
if not PREVIOUS_OUTPUT:
    raise RuntimeError('Copia la ruta neorx-run de la salida de la cuarta sesión en PREVIOUS_OUTPUT.')
OUTPUT = WORK / 'neorx-evaluation'
args = [sys.executable, '-u', str(script_path), '--bundle', str(BUNDLE),
        '--previous-output', PREVIOUS_OUTPUT, '--output', str(OUTPUT), '--device', 'cpu', '--cpu-threads', '2']
if DATA_ROOT:
    args += ['--data-root', DATA_ROOT]
if CLOSE_BY_BUDGET:
    args += ['--close-by-budget']
log_path = WORK / 'evaluation_console.log'
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(args, cwd=BUNDLE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if OUTPUT.exists():
    shutil.copy2(log_path, OUTPUT / 'console.log')
    from training.kaggle_run import package_outputs
    package_outputs(OUTPUT)
if return_code:
    raise RuntimeError('Evaluación incompleta: revisa el log y STATUS.json.')

In [ ]:
from IPython.display import display, FileLink
print((OUTPUT / 'STATUS.json').read_text(encoding='utf-8'))
print('Descarga el ZIP y revisa evaluation/metrics.csv, evaluation/routing_summary.json y FREEZE.json.')
print('Una evaluación correcta del software no garantiza resultados clínicos adecuados.')
display(FileLink('/kaggle/working/neorx_entrenamiento_resultados.zip'))